# FAO DATALAB

Objectif : produire une analyse détaillée sur les données de la FAO de 2023 sur la sous-nutrition à l'échelle mondiale

## Initialisation du projet

Importation des librairies

In [76]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
sns.set_theme(style="whitegrid")

Importation des datasets

*Import de 5 fichiers CSV de FAOSTAT tous structurés selon le schéma standard FAO : Code Domaine, Domaine, Code zone, Zone, Code Élément, Élément, Code Produit, Produit, Code année, Année, Unité, Valeur, Note*

In [77]:
df_animaux = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_animaux.csv")
df_cereales = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_céréales.csv")
df_population = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_population.csv")
df_vegetaux = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\2023\fr_vegetaux.csv")
df_sousalimentation = pd.read_csv(r"C:\Users\maril\Documents\FAO_Datalab\Data\Périodes_glissantes\fr_sousalimentation.csv")

Vérification de l'importation - Encodage

In [78]:
df_animaux.head()
df_cereales.head()
df_population.head()
df_vegetaux.head()
df_sousalimentation.head()

,Code Domaine,Domaine,Code zone (FAO),Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20192021,2019-2021,%,28.9,E,Valeur estimée,NaN
1,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20202022,2020-2022,%,31,E,Valeur estimée,NaN
2,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20212023,2021-2023,%,32,E,Valeur estimée,NaN
3,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20222024,2022-2024,%,29.7,E,Valeur estimée,NaN
4,FS,Données de la sécurité alimentaire,2,Afghanistan,6121,Valeur,210041,Prévalence de la sous-alimentation (%) (moyenne sur 3 ans),20232025,2023-2025,%,27.8,E,Valeur estimée,NaN


Visualisation des dimensions

*La visualisation est faite pour rendre compte de la structure de chaque fichier*

In [79]:
print(f"vegetaux          | {df_vegetaux.shape[0]:6} lignes | {df_vegetaux.shape[1]} colonnes")
print(f"animaux           | {df_animaux.shape[0]:6} lignes | {df_animaux.shape[1]} colonnes")
print(f"cereales          | {df_cereales.shape[0]:6} lignes | {df_cereales.shape[1]} colonnes")
print(f"population        | {df_population.shape[0]:6} lignes | {df_population.shape[1]} colonnes")
print(f"sousalimentation  | {df_sousalimentation.shape[0]:6} lignes | {df_sousalimentation.shape[1]} colonnes")

vegetaux          |  91617 lignes | 15 colonnes
animaux           |  33432 lignes | 15 colonnes
cereales          |  14203 lignes | 15 colonnes
population        |    177 lignes | 15 colonnes
sousalimentation  |   2040 lignes | 15 colonnes


**Constat :** les 5 datasets de la FAO issus de FAOSTATS pour l'année 2023 sont formatés de la même manière : les 15 colonnes du schéma standard susmentionné

# Etape 1 : Diagnostic qualité et préparation des données

## A. Diagnostic : Audit technique et POO
*Présentation de la classe 'DataProfiler' et 'ProfileurFAO'*

**Classe Mère**

*Il s'agit d'une classe générique permettant qu'être utilisée pour tous autres fichiers indépendemment du sujet traité dans ce projet*


In [80]:
class DataProfiler:
    """
    Classe mère générique de diagnostic qualité pour tout type de dataset.
    Permet de centraliser les contrôles structurels de base (taille, NaN, doublons).
    """
    
    def __init__(self, df: pd.DataFrame, nom_source: str):
        self.df = df
        self.nom_source = nom_source

    def rapport_nan_global(self) -> pd.DataFrame:
        """
        Calcul du nombre et du pourcentage de valeurs manquantes pour chaque colonne.
        Paramètres : Aucun
        Retourne : pd.Dataframe -> Un tableau synthétique listant les colonnes, le nb de NaN et leur taux en %.
        """
        total_lignes = len(self.df)
        nan_count = self.df.isna().sum()
        nan_pct = (nan_count / total_lignes * 100) if total_lignes > 0 else nan_count * 0
        
        df_synthese = pd.DataFrame({
            "Colonne": nan_count.index,
            "Nb_NaN": nan_count.values,
            "Taux_Pct": nan_pct.values.round(2)
        })
        return df_synthese[df_synthese["Nb_NaN"] > 0].reset_index(drop=True)

    def rapport_doublons(self, subset=None) -> int:
        """
        Identification et comptage du nombre de lignes dupliquées.
        Paramètres : subset (list ou str, optionnel) -> Colonnes à considérer pour la recherche de doublons.
        Retourne : int -> Le nombre total de doublons stricts ou sur le subset choisi.
        """
        return int(self.df.duplicated(subset=subset).sum())

    def detecter_valeurs_atypiques(self, colonne: str, seuil_min=None, seuil_max=None) -> pd.DataFrame:
        """
        Isolement des lignes sortant de plages plausibles métier (ex: valeurs aberrantes de disponibilité).
        Paramètres :
        colonne : str -> Nom de la colonne à auditer.
        seuil_min : float, optionnel -> Seuil minimal acceptable.
        seuil_max : float, optionnel -> Seuil maximal acceptable.
        Retourne : pd.DataFrame -> Sous-ensemble du DataFrame contenant uniquement les lignes atypiques.
        """
        if colonne not in self.df.columns:
            return pd.DataFrame()
        
        s=pd.to_numeric(self.df[colonne], errors='coerce')

        masque = pd.Series([False] * len(self.df), index=self.df.index)
        if seuil_min is not None:
            masque = masque | (s < seuil_min)
        if seuil_max is not None:
            masque = masque | (s > seuil_max)

        return self.df[masque]
    
    def zones_uniques(self) -> set:
        """
        Retourne l'ensemble des zones (pays) uniques présentes dans la source.
        Paramètres : Aucun
        Retourne : set -> Ensemble des zones géographiques.
        """
        if "Zone" in self.df.columns:
            return set(self.df["Zone"].dropna().unique())
        return set()

    def resume_audit(self) -> dict:
        """
        Génèration d'un dictionnaire de synthèse globale de la source.
        Évite les prints bruts pour faciliter l'intégration dans un tableau de bord ou un DataFrame.
        Paramètres : Aucun
        Retourne : dict -> Dictionnaire contenant le nom de la source, le nombre de lignes et de colonnes.
        """
        return {
            "Source": self.nom_source,
            "Nb_Lignes": len(self.df),
            "Nb_Colonnes": len(self.df.columns)
        }


**Classe Fille**

*Classe spécifique à notre dataset qui va aller chercher les spécificités de nos jeux de données*

In [81]:
class ProfileurFAO(DataProfiler):
    """
    Classe fille spécialisée pour auditer les spécificités des fichiers FAO 
    (gestion des clés de jointure géographiques et des seuils nutritionnels).
    """
    
    CLE_JOINTURE = "Zone"

    def verifier_coherence_cles(self, autre_df: pd.DataFrame) -> set:
        """
        Identification des pays présents dans la table courante mais absents d'une autre table de référence.
        Paramètres : autre_df : pd.DataFrame -> Le DataFrame avec lequel comparer les zones géographiques.
        Retourne : set -> L'ensemble des zones (pays) en rupture de jointure.
        """
        zones_source = set(self.df[self.CLE_JOINTURE].dropna())
        zones_autre = set(autre_df[self.CLE_JOINTURE].dropna())
        return zones_source - zones_autre

    def recenser_unites(self) -> pd.DataFrame:
        """
        Recensemment des couples uniques entre les libellés d'Élément et leurs Unités de mesure.
        Paramètres : Aucun    
        Retourne : pd.DataFrame -> Un tableau listant les combinaisons Élément / Unité présentes dans le dataset.
        """
        if 'Élément' in self.df.columns and 'Unité' in self.df.columns:
            return self.df[['Élément', 'Unité']].drop_duplicates().reset_index(drop=True)
        return pd.DataFrame()

    # Dictionnaire de correspondance exhaustif des symboles FAO
    DICT_SYMBOLES_FAO = {
        'A': 'Donnée officielle (Official)',
        'E': 'Donnée estimée (Estimated)',
        'I': 'Donnée imputée par la FAO (Imputed)',
        'M': 'Donnée manquante (Missing)',
        'O': 'Valeur manquante (Missing value)',
        'Q': 'Valeur manquante ou masquée (Missing/Suppressed)',
        'S': 'Donnée non officielle / agrégée',
        'X': 'Source externe / Organisation internationale',
        '*': 'Donnée non officielle',
        'Fc': 'Donnée calculée (Calculated)'
    }

    def analyser_fiabilite_symboles(self) -> pd.DataFrame:
        """
        Analyse de la répartition statistique de la colonne Symbole pour évaluer la fiabilité des données.
        Paramètres : Aucun  
        Retourne : pd.DataFrame -> Un tableau croisé indiquant l'effectif et le pourcentage de chaque symbole de fiabilité.
        """
        if 'Symbole' in self.df.columns:
            total = len(self.df)
            counts = self.df['Symbole'].value_counts()
            pct = (counts / total * 100).round(2)

            df_sym = pd.DataFrame({"Effectif": counts, "Part_Pct": pct}).reset_index()
            df_sym.columns = ["Symbole", "Effectif", "Part_Pct"]
            
            # Ajout de la colonne explicative
            df_sym["Signification"] = df_sym["Symbole"].map(self.DICT_SYMBOLES_FAO).fillna("Donnée officielle / Non renseigné")
            
            return df_sym
        return pd.DataFrame()

**Exécution des méthodes de profiling sur les 5 fichiers sources**

*Objectif : Appliquer nos méthodes sur l'ensemble des CSV afin d'observer les spécificités, les variations ou encore les données "suspectes"*

In [82]:
# STOCKAGE des 5 DataFrames et leurs noms dans un dictionnaire pour les parcourir facilement
fichiers_dict = {
    "Animaux": df_animaux,
    "Céréales": df_cereales,
    "Population": df_population,
    "Végétaux": df_vegetaux,
    "Sous-alimentation": df_sousalimentation
}

print("##################################################")
print("### PARTIE A : CONTRÔLES STRUCTURELS           ###")
print("##################################################\n")

# --- 1. TABLEAU DE SYNTHÈSE GLOBALE DE L'AUDIT (Dimensions) ---
synthese_globale = []

for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    synthese_globale.append(profiler.resume_audit())

df_synthese_globale = pd.DataFrame(synthese_globale)
print("\n=== 1. SYNTHÈSE GLOBALE DE L'AUDIT (Dimensions) ===")
display(df_synthese_globale)

# --- 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ---
synthese_nan = []

for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    df_nan = profiler.rapport_nan_global()
    # On filtre uniquement les colonnes qui ont des NaN pour ne garder que l'essentiel
    df_nan_filtré = df_nan[df_nan["Nb_NaN"] > 0].copy()
    if not df_nan_filtré.empty:
        df_nan_filtré["Source"] = nom
        synthese_nan.append(df_nan_filtré)

if synthese_nan:
    df_synthese_nan = pd.concat(synthese_nan, ignore_index=True)
    df_synthese_nan = df_synthese_nan[["Source", "Colonne", "Nb_NaN", "Taux_Pct"]]
    print("=== 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ===")
    display(df_synthese_nan)
else:
    print("=== 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ===")
    print("Aucune valeur manquante détectée dans les fichiers !")


# --- 3. RAPPORT DES DOUBLONS ---
synthese_doublons = []
for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    
    # Doublons stricts (sur toute la ligne)
    doublons_stricts = profiler.rapport_doublons(subset=None)
    
    # Doublons sur clé métier (Zone, Élément, Produit si disponible)
    colonnes_dispo = [col for col in ['Zone', 'Élément', 'Produit'] if col in df.columns]
    doublons_metier = profiler.rapport_doublons(subset=colonnes_dispo) if len(colonnes_dispo) > 1 else 0

    synthese_doublons.append({
        "Source": nom,
        "Doublons_Stricts": doublons_stricts,
        "Doublons_Cle_Metier": doublons_metier
    })

df_synthese_doublons = pd.DataFrame(synthese_doublons)
print("\n=== 3. RAPPORT DES DOUBLONS (Stricts et Clé Métier) ===")
display(df_synthese_doublons)

pd.set_option('display.max_colwidth', None)
# --- 4. VALEURS NÉGATIVES / ATYPIQUES (Détail par Colonne et Élément) ---
print("\n=== 4. RAPPORT DES VALEURS NÉGATIVES / ATYPIQUES ===")

synthese_atypiques = []

for nom, df in fichiers_dict.items():
    profiler = DataProfiler(df, nom)
    
    # Isolation des lignes où Valeur < 0
    df_neg = profiler.detecter_valeurs_atypiques(colonne='Valeur', seuil_min=0)
    
    if not df_neg.empty:
        # Récupération de la valeur minimale
        val_min = pd.to_numeric(df_neg['Valeur'], errors='coerce').min()
        
        # Décompte et formatage des éléments impactés (ex: "Variation de stock (200), Disponibilité intérieure (109)")
        if 'Élément' in df_neg.columns:
            counts = df_neg['Élément'].value_counts()
            elements_str = ", ".join([f"{elt} ({count})" for elt, count in counts.items()])
        else:
            elements_str = "Non spécifié"
            
        synthese_atypiques.append({
            "Source": nom,
            "Colonne": "Valeur",
            "Nb_Valeurs_Négatives": len(df_neg),
            "Valeur_Min": val_min,
            "Répartition_Éléments": elements_str
        })
    else:
        synthese_atypiques.append({
            "Source": nom,
            "Colonne": "Valeur",
            "Nb_Valeurs_Négatives": 0,
            "Valeur_Min": "-",
            "Répartition_Éléments": "Aucune"
        })

df_synthese_atypiques = pd.DataFrame(synthese_atypiques)
display(df_synthese_atypiques)

##################################################
### PARTIE A : CONTRÔLES STRUCTURELS           ###
##################################################


=== 1. SYNTHÈSE GLOBALE DE L'AUDIT (Dimensions) ===


,Source,Nb_Lignes,Nb_Colonnes
0,Animaux,33432,15
1,Céréales,14203,15
2,Population,177,15
3,Végétaux,91617,15
4,Sous-alimentation,2040,15


=== 2. RAPPORT GLOBAL DES VALEURS MANQUANTES ===


,Source,Colonne,Nb_NaN,Taux_Pct
0,Animaux,Note,33432,100.00
1,Céréales,Note,14203,100.00
2,Population,Note,177,100.00
3,Végétaux,Note,91617,100.00
4,Sous-alimentation,Valeur,668,32.75
5,Sous-alimentation,Note,2040,100.00



=== 3. RAPPORT DES DOUBLONS (Stricts et Clé Métier) ===


,Source,Doublons_Stricts,Doublons_Cle_Metier
0,Animaux,0,0
1,Céréales,0,0
2,Population,0,0
3,Végétaux,0,0
4,Sous-alimentation,0,1632



=== 4. RAPPORT DES VALEURS NÉGATIVES / ATYPIQUES ===


,Source,Colonne,Nb_Valeurs_Négatives,Valeur_Min,Répartition_Éléments
0,Animaux,Valeur,129,-1107.0,"Variation de stock (109), Disponibilité intérieure (20)"
1,Céréales,Valeur,309,-5530.0,"Variation de stock (291), Disponibilité intérieure (18)"
2,Population,Valeur,0,-,Aucune
3,Végétaux,Valeur,243,-1907.0,Disponibilité intérieure (243)
4,Sous-alimentation,Valeur,0,-,Aucune


**Constat :** Les 5 fichiers présentent une structure cohérente et standardisée de 15 colonnes par table. La volumétrie globale est très variable (de 177 lignes pour Population à 91 617 lignes pour Végétaux), ce qui reflète la diversité du niveau de détail entre les bilans de masse volumineux et les données démographiques agrégées.

Colonne Note : Présente un taux de vacuité absolu de 100 % sur l'intégralité des 5 fichiers. Il s'agit d'une colonne système résiduelle inutilisée lors de l'export FAO, qui pourra être purement et simplement supprimée lors du nettoyage.

Colonne Valeur (Sous-alimentation) : 668 valeurs sont manquantes, soit 32,75 % de la table. Cette vacuité n'est pas un défaut d'extraction technique : la FAO ne publie pas de chiffres de sous-alimentation pour les très petits territoires ou lorsque la prévalence est estimée en dessous du seuil de détection statistique (souvent noté <2,5 % ou omis dans les publications officielles).

Doublons stricts : Aucun doublon parfait n'est détecté sur l'ensemble de la base (0 sur toutes les tables), ce qui garantit l'intégrité technique de l'importation.

Doublons sur clé métier (Zone, Élément, Produit) : Parfaitement nuls sur les bilans alimentaires et la population. En revanche, 1 632 doublons de clé apparaissent dans la table Sous-alimentation. Cela est tout à fait normal : cette table suit une logique temporelle par périodes triennales glissantes (ex: 2012-2014, 2013-2015) et contient plusieurs indicateurs. Sans l'intégration de la colonne Année/Période dans la clé primaire, ces lignes apparaissent logiquement comme répétitives.

L'audit révèle 681 valeurs négatives concentrées exclusivement sur les trois fichiers de bilans alimentaires (309 dans Céréales, 243 dans Végétaux et 129 dans Animaux, avec un minimum extrême à -5 530), tandis que Population et Sous-alimentation en sont exempts. Sur le plan métier, ces données se divisent en deux catégories : 400 lignes de « Variation de stock » négatives (majoritaires dans Céréales et Animaux), qui sont parfaitement légitimes et traduisent un déstockage net sur la période, et 281 lignes de « Disponibilité intérieure » négatives (100 % des cas de Végétaux, ainsi que quelques lignes dans Animaux et Céréales), qui découlent d'un déséquilibre déclaratif dans l'équation du bilan de masse FAO. En phase de nettoyage, les variations de stock pourront être conservées intactes pour le calcul des ressources, alors que les disponibilités intérieures négatives devront faire l'objet d'un retraitement spécifique (recalcul ou ajustement à zéro).




In [84]:
print("##################################################")
print("### PARTIE B : SPÉCIFICITÉS FAO                ###")
print("##################################################\n")

# --- ANALYSE SPECIFIQUE : NaN par période (sous-alimentation) ---
nan_sousalim = df_sousalimentation[df_sousalimentation['Valeur'].isna()]
colonne_periode = 'Année' if 'Année' in df_sousalimentation.columns else 'Période'
repartition_nan_periode = nan_sousalim[colonne_periode].value_counts().reset_index()
repartition_nan_periode.columns = [colonne_periode, 'Nombre_NaN']

print(f"\n===  ANALYSE SPECIFIQUE :  NaN PAR PÉRIODE (Sous-alimentation) ===")
print(f"Total des valeurs manquantes : {len(nan_sousalim)}")
display(repartition_nan_periode)

# --- COUVERTURE GÉOGRAPHIQUE CROISÉE ENTRE TOUS LES FICHIERS (via zones_uniques et cohérence) ---
print("\n=== COUVERTURE GÉOGRAPHIQUE CROISÉE (Nombre de zones uniques par fichier) ===")
# Ensemble complet de tous les pays uniques (tous fichiers confondus)
tous_les_pays = set().union(*[ProfileurFAO(df, nom).zones_uniques() for nom, df in fichiers_dict.items()])

# Tableau de synthèse des écarts
synthese_ecarts = []
for nom, df in fichiers_dict.items():
    profiler = ProfileurFAO(df, nom)
    zones = profiler.zones_uniques()
    manquants = sorted(list(tous_les_pays - zones))
    
    synthese_ecarts.append({
        "Source": nom,
        "Nb_Zones": len(zones),
        "Nb_Manquants": len(manquants),
    })

zones_sousalim = ProfileurFAO(df_sousalimentation, "Sous-alimentation").zones_uniques()


fichiers_autres = [df_animaux, df_cereales, df_population, df_vegetaux]
zones_autres = set().union(*[ProfileurFAO(df, "").zones_uniques() for df in fichiers_autres])
pays_exclusifs = zones_sousalim - zones_autres
print(f"{len(pays_exclusifs)} pays présents dans sousalimentation mais absents des autres fichiers :\n")
for pays in sorted(pays_exclusifs):
    print(f" - {pays}")

display(pd.DataFrame(synthese_ecarts))

# --- RECENSSEMENT DES UNITÉS DE MESURE PAR FICHIER ---
print("\n=== RECENSSEMENT DES UNITÉS DE MESURE ===")
for nom, df in fichiers_dict.items():
    profiler = ProfileurFAO(df, nom)
    df_unites = profiler.recenser_unites()
    if not df_unites.empty:
        print(f"\n--- Unités pour la source : {nom} ---")
        display(df_unites)


# --- FIABILITÉ DE LA DONNÉE (Répartition des Symboles) ---
print("\n=== FIABILITÉ DE LA DONNÉE (Répartition de la colonne Symbole) ===")
for nom, df in fichiers_dict.items():
    profiler = ProfileurFAO(df, nom)
    df_sym = profiler.analyser_fiabilite_symboles()
    if not df_sym.empty:
        print(f"\n--- Symboles pour la source : {nom} ---")
        display(df_sym)

##################################################
### PARTIE B : SPÉCIFICITÉS FAO                ###
##################################################


===  ANALYSE SPECIFIQUE :  NaN PAR PÉRIODE (Sous-alimentation) ===
Total des valeurs manquantes : 668


,Année,Nombre_NaN
0,2023-2025,139
1,2022-2024,136
2,2021-2023,134
3,2020-2022,133
4,2019-2021,126



=== COUVERTURE GÉOGRAPHIQUE CROISÉE (Nombre de zones uniques par fichier) ===
27 pays présents dans sousalimentation mais absents des autres fichiers :

 - Andorre
 - Bermudes
 - Brunéi Darussalam
 - Burundi
 - Bénin
 - Cuba
 - Dominique
 - Groenland
 - Guinée équatoriale
 - Japon
 - Mali
 - Nioué
 - Palaos
 - Palestine
 - Porto Rico
 - République centrafricaine
 - République populaire démocratique de Corée
 - Samoa américaines
 - Singapour
 - Somalie
 - Soudan
 - Soudan du Sud
 - Tchad
 - Togo
 - Tokélaou
 - Érythrée
 - Îles Cook


,Source,Nb_Zones,Nb_Manquants
0,Animaux,177,27
1,Céréales,177,27
2,Population,177,27
3,Végétaux,177,27
4,Sous-alimentation,204,0



=== RECENSSEMENT DES UNITÉS DE MESURE ===

--- Unités pour la source : Animaux ---


...

,Symbole,Effectif,Part_Pct,Signification
0,X,177,100.0,Source externe / Organisation internationale



--- Symboles pour la source : Végétaux ---


,Symbole,Effectif,Part_Pct,Signification
0,I,56516,61.69,Donnée imputée par la FAO (Imputed)
1,E,35101,38.31,Donnée estimée (Estimated)



--- Symboles pour la source : Sous-alimentation ---


,Symbole,Effectif,Part_Pct,Signification
0,E,1372,67.25,Donnée estimée (Estimated)
1,O,360,17.65,Valeur manquante (Missing value)
2,Q,308,15.10,Valeur manquante ou masquée (Missing/Suppressed)


**Constat spécifique aux données de la FAO** : 

Couverture géographique : L'analyse croisée révèle un socle commun de 177 pays sur les 4 fichiers principaux (Animaux, Céréales, Végétaux, Population), tandis que la table Sous-alimentation s'étend sur 204 zones (soit 27 pays exclusifs tels que le Japon, Singapour ou la Corée du Nord). Les 668 valeurs manquantes identifiées dans la sous-alimentation (réparties de manière stable à environ 130 NaNs par période) s'expliquent par cette différence de champ géographique et par des restrictions de diffusion de la FAO sur certaines zones. On note également la possibilité de manque de registres douaniers et agricoles exploitatbles pour bâtir des bilans de masse du fait de confits, d'isolement politiques ou de statuts de micro-territoires. Ce décalage n'est pas une anomalie de données, mais une limite méthodologique d'origine : les jointures d'analyse devront se restreindre à l'intersection des 177 pays pour garantir des bilans cohérents.

Harmonisation des unités de mesure : Les unités de mesure sont parfaitement standardisées par type d'indicateur (1000 t pour les flux physiques, 1000 No pour les effectifs de population, kcal/personne/jour et g/personne/jour pour les apports nutritionnels). Il n'y a pas d'incohérence de libellé au sein des tables, mais leur présence sous des ordres de grandeur distincts impose une règle de gestion stricte lors du nettoyage : des conversions systématiques seront obligatoires avant de croiser les tables pour calculer des ratios mondiaux exacts.

Qualité et fiabilité des données : La table Population repose à 100 % sur des données externes (symbole X), tandis que les trois fichiers de bilans alimentaires ne contiennent aucune donnée officielle brute, se partageant uniquement entre données estimées (E) et données imputées par la FAO (I).ette bascule entre estimation et imputation établit un gradient de confiance : les bilans végétaux et céréaliers intègrent une incertitude statistique plus élevée que les bilans animaux. Cette variabilité devra être prise en compte pour nuancer les analyses sur les pays dont les données sont très fortement imputées.

## B. Rapport d'anomalies quantifié

**Présentation synthétique et chiffré des anomalies détectées permettant une aide à la prise de décisions stratégiques en lien avec le contexte du projet**

In [85]:
import pandas as pd
from IPython.display import display, HTML

# --- RAPPORT D'ANOMALIE QUANTIFIÉ ET SYNTHÈSE DIAGNOSTIC ---
synthese = pd.DataFrame([
    {
        "Point de diagnostic": "Valeurs manquantes",
        "Résultat": "668 NaN uniquement dans Sous-alimentation (~130 par période, 0 dans les 4 autres fichiers)",
        "Nature": "Structurel FAO — Incomplétude liée aux zones de crise, régimes fermés et masquage de données"
    },
    {
        "Point de diagnostic": "Doublons",
        "Résultat": "0 doublon détecté (ligne complète et clé métier), sur l'ensemble des 5 fichiers",
        "Nature": "Aucune action requise"
    },
    {
        "Point de diagnostic": "Couverture géographique",
        "Résultat": "177 pays (Veg/Ani/Cer/Pop) vs 204 (Sous-alimentation) -> 27 pays perdus à la jointure",
        "Nature": "Limite méthodologique (micro-territoires & régimes fermés) — Choix de jointure restreinte au socle des 177 pays"
    },
    {
        "Point de diagnostic": "Unités de mesure",
        "Résultat": "5 unités mélangées ('1000 t', '1000 No', 'kcal/personne/jour', 'g/personne/jour', 'kg/personne/an')",
        "Nature": "Point de vigilance technique — Conversions explicites obligatoires (x1 000 pop, x1 000 000 kg) avant aggregation"
    },
    {
        "Point de diagnostic": "Fiabilité (Symbole)",
        "Résultat": "0% de données brutes officielles : 100% X (Pop), ~61.5% I (Végétaux/Céréales), 61.8% E (Animaux)",
        "Nature": "Axe de lecture qualité — Gradient de confiance (plus forte incertitude sur le végétal/céréales) à conserver"
    },
    {
        "Point de diagnostic": "Valeurs négatives / extrêmes",
        "Résultat": "681 valeurs négatives (309 Céréales, 243 Végétaux, 129 Animaux) : 400 'Variation de stock', 281 'Disponibilité intérieure'",
        "Nature": "400 déstockages légitimes (à conserver) ; 281 anomalies comptables déclaratives (à recalculer ou imputer à 0)"
    },
    {
        "Point de diagnostic": "Particularité métier (Formatage)",
        "Résultat": "Présence de la notation textuelle '<0.1' (ou valeurs seuils) dans la colonne Valeur de Sous-alimentation",
        "Nature": "Spécificité métier FAO — Impossibilité de conversion float directe ; décision de traitement : imputation à 0.05 ou 0.1"
    }
])


pd.set_option('display.max_colwidth', None)

# Affichage sous forme de tableau propre
display(HTML(synthese.to_html(index=False)))

Point de diagnostic,Résultat,Nature
Valeurs manquantes,"668 NaN uniquement dans Sous-alimentation (~130 par période, 0 dans les 4 autres fichiers)","Structurel FAO — Incomplétude liée aux zones de crise, régimes fermés et masquage de données"
Doublons,"0 doublon détecté (ligne complète et clé métier), sur l'ensemble des 5 fichiers",Aucune action requise
Couverture géographique,177 pays (Veg/Ani/Cer/Pop) vs 204 (Sous-alimentation) -> 27 pays perdus à la jointure,Limite méthodologique (micro-territoires & régimes fermés) — Choix de jointure restreinte au socle des 177 pays
Unités de mesure,"5 unités mélangées ('1000 t', '1000 No', 'kcal/personne/jour', 'g/personne/jour', 'kg/personne/an')","Point de vigilance technique — Conversions explicites obligatoires (x1 000 pop, x1 000 000 kg) avant aggregation"
Fiabilité (Symbole),"0% de données brutes officielles : 100% X (Pop), ~61.5% I (Végétaux/Céréales), 61.8% E (Animaux)",Axe de lecture qualité — Gradient de confiance (plus forte incertitude sur le végétal/céréales) à conserver
Valeurs négatives / extrêmes,"681 valeurs négatives (309 Céréales, 243 Végétaux, 129 Animaux) : 400 'Variation de stock', 281 'Disponibilité intérieure'",400 déstockages légitimes (à conserver) ; 281 anomalies comptables déclaratives (à recalculer ou imputer à 0)
Particularité métier (Formatage),Présence de la notation textuelle '<0.1' (ou valeurs seuils) dans la colonne Valeur de Sous-alimentation,Spécificité métier FAO — Impossibilité de conversion float directe ; décision de traitement : imputation à 0.05 ou 0.1


**Points de vigilance**

Périmètre géographique & Jointure : La jointure du dataset global devra explicitement documenter et justifier le sort des 27 pays de la table Sous-alimentation absents des trois bilans alimentaires et de la population (jointure interne assumée sur les 177 pays communs, avec la liste des 27 pays exclus fournie en annexe/documentation). La source officile FAOSTATs précise : **La comparabilité géographique est limitée en raison des différences de méthodes et de champ d'application, sauf pour les régions composées de pays homogènes**.

Identification des indicateurs : Le calcul des indicateurs (kcal/personne/jour, g/personne/jour, taux de sous-nutrition) devra obligatoirement s'appuyer sur le libellé complet de la colonne Élément et non sur la seule colonne Unité, afin d'éviter toute confusion entre les flux physiques (1000 t) et les ratios nutritionnels.

Traitement des valeurs négatives : Les 400 valeurs négatives associées à la « Variation de stock » doivent être conservées telles quelles (légitimité physique traduisant un déstockage net), tandis que les 281 valeurs négatives de « Disponibilité intérieure » devront faire l'objet d'un retraitement spécifique (recalcul ou mise à zéro).

Gestion des seuils de sous-alimentation : Les valeurs textuelles de type "<0.1" dans la table Sous-alimentation doivent être converties numériquement de façon explicite (ex: imputation à 0.05 ou 0.1 selon la règle de gestion choisie) avant toute agrégation ou calcul de ratio.

Isolation temporelle : La période d'analyse de référence devra être filtrée/isolée dès la première étape de préparation, avant toute opération de jointure entre les fichiers, pour éviter la multiplication indésirable de lignes.

## C. Nettoyage, stratégies d'imputation et fusion des sources
* **Décisions de traitement et justifications :** - Choix de jointure entre les fichiers végétaux/animaux/population et justification de l'exclusion ou du traitement de certaines valeurs aberrantes (ex: plafonnement pour la Dominique).
* Constitution du dataset global (`df_fao_global.csv`).

# Etape 2 : Analyse exploratoire 

# Etape 3 : Modelisation : régression 

# Etape 4 : Clustering et restitution